### Ground Truth Annotation Template Generation
This notebook creates the 120 candidate-job pairs for evaluation and exports a blank annotation template CSV for manual scoring.
- Create 120 candidate-job pairs (4 jobs for each of the 30 candidates) and export a blank CSV template to manually assign scores for candidate.


In [ ]:
import os, sys, re
from pathlib import Path
import numpy as np
import pandas as pd

if hasattr(sys.stdout, "reconfigure"):
    try: sys.stdout.reconfigure(encoding='utf-8')
    except: pass

RANDOM_STATE = 42
PROJECT_ROOT = Path("c:/Users/yzhoo/AI-Agent-Job-Search")
PROCESSED_DIR = PROJECT_ROOT / "experiments" / "dataset" / "processed"

# Load the sampled resumes and jobs
resumes_df = pd.read_csv(PROCESSED_DIR / "benchmark_resumes.csv")
jobs_df = pd.read_csv(PROCESSED_DIR / "benchmark_jobs.csv")

print(f"Loaded {len(resumes_df)} candidates and {len(jobs_df)} jobs.")
print("Candidates by domain:")
print(resumes_df['category'].value_counts().to_string())
print("\nJobs by domain:")
print(jobs_df['category'].value_counts().to_string())
print("\nJobs salary disclosure:")
print(jobs_df.groupby(['category', 'is_salary_disclosed']).size().unstack(fill_value=0))


Loaded 30 candidates and 60 jobs.
Candidates by domain:
category
INFORMATION-TECHNOLOGY    20
ACCOUNTANT                10

Jobs by domain:
category
Information & Communication Technology    40
Accounting                                20

Jobs salary disclosure:
is_salary_disclosed                     False  True 
category                                            
Accounting                                 10     10
Information & Communication Technology     20     20


### 1. Candidate-Job Pairing

Assign 4 jobs to each of the 30 candidates.



In [2]:
# Candidate-job pairing plan (4 jobs per candidate)
it_pairs_plan = {
    "C01": ["J15", "J14", "J21", "J41"],
    "C02": ["J27", "J16", "J01", "J42"],
    "C03": ["J12", "J09", "J05", "J43"],
    "C04": ["J07", "J18", "J19", "J44"],
    "C05": ["J23", "J02", "J37", "J45"],
    "C06": ["J32", "J11", "J08", "J46"],
    "C07": ["J25", "J18", "J35", "J47"],
    "C08": ["J28", "J23", "J06", "J48"],
    "C09": ["J14", "J32", "J31", "J49"],
    "C10": ["J30", "J40", "J13", "J50"],
    "C11": ["J16", "J26", "J38", "J51"],
    "C12": ["J08", "J04", "J33", "J52"],
    "C13": ["J40", "J07", "J36", "J53"],
    "C14": ["J22", "J29", "J04", "J54"],
    "C15": ["J02", "J39", "J17", "J55"],
    "C16": ["J26", "J13", "J24", "J56"],
    "C17": ["J11", "J25", "J34", "J57"],
    "C18": ["J03", "J27", "J39", "J58"],
    "C19": ["J16", "J05", "J10", "J59"],
    "C20": ["J20", "J03", "J18", "J60"],
}

acc_pairs_plan = {
    "C21": ["J48", "J43", "J44", "J01"],
    "C22": ["J55", "J41", "J50", "J02"],
    "C23": ["J59", "J54", "J46", "J03"],
    "C24": ["J45", "J57", "J53", "J05"],
    "C25": ["J60", "J49", "J47", "J07"],
    "C26": ["J51", "J58", "J42", "J09"],
    "C27": ["J52", "J55", "J56", "J12"],
    "C28": ["J53", "J48", "J44", "J15"],
    "C29": ["J57", "J52", "J49", "J16"],
    "C30": ["J43", "J60", "J41", "J23"],
}

all_pairs = []
pair_id_counter = 1

for plan in (it_pairs_plan, acc_pairs_plan):
    for candidate_id, job_ids in plan.items():
        for job_id in job_ids:
            all_pairs.append({
                "pair_id": f"P{pair_id_counter:03d}",
                "candidate_id": candidate_id,
                "job_id_code": job_id,
            })
            pair_id_counter += 1

pairs_df = pd.DataFrame(all_pairs)

print(f"Total pairs generated: {len(pairs_df)}")
print(f"Candidates:           {pairs_df['candidate_id'].nunique()} (30 candidates)")
print(f"Jobs covered:         {pairs_df['job_id_code'].nunique()} / 60")
print(f"Duplicate pairs:      {pairs_df.duplicated(subset=['candidate_id', 'job_id_code']).sum()}")


Total pairs generated: 120
Candidates:           30 (30 candidates)
Jobs covered:         60 / 60
Duplicate pairs:      0


### 2. Generate Candidate/Job Summaries & Template CSV

To make manual scoring easier:
- `candidate_summary`: Combines role, experience, education, location, salary, and skills into one line.
- `job_summary`: Combines job title, company, requirements, location, salary, and job type into one line.
- Scoring columns (`score_title`, `score_location`, `score_education`, `score_type`, `score_salary`, `score_experience`, `ground_truth_score`, `relevance_grade`, `annotation_rationale`) are left blank for manual scoring.


In [3]:
# Education requirements from job descriptions
job_edu_requirements = {
    # IT Jobs (J01 - J40)
    "J01": "Not Specified",
    "J02": "Bachelor's Degree or Professional Degree in Computer Science, Information Technology, or equivalent",
    "J03": "Not Specified",
    "J04": "Not Specified",
    "J05": "Not Specified",
    "J06": "Bachelor's, Master's, or Ph.D. in Software Engineering, Computer Engineering, Computer Science, or Electrical & Electronics Engineering",
    "J07": "Not Specified",
    "J08": "Diploma in Computer Science, Information Technology, Engineering (Computer/Telecommunication), or equivalent",
    "J09": "Diploma, Advanced/Graduate Diploma, Bachelor's Degree, Post Graduate Diploma, or Professional Degree",
    "J10": "Bachelor's Degree in IT, Computer Science, Computer Engineering, or equivalent",
    "J11": "University Degree in Computer Science, Electronics, Information Systems, Business Administration, or related field",
    "J12": "Bachelor's in Computer Science or IT with SAP B1/HANA and Microsoft Engineer/Professional Certifications",
    "J13": "Diploma, Advanced/Graduate Diploma, Bachelor's Degree, Post Graduate Diploma, or Professional Degree in IT or equivalent",
    "J14": "Not Specified",
    "J15": "Not Specified",
    "J16": "Bachelor's degree in Computer Science, Information Technology, or a related field",
    "J17": "Not Specified",
    "J18": "Degree in Computer Science, Information Technology, or equivalent",
    "J19": "Bachelor's Degree, Post Graduate Diploma, or Professional Degree in Computer Science, Information Technology, or equivalent",
    "J20": "Not Specified",
    "J21": "BSc Degree in Computer Science or relevant field",
    "J22": "Bachelor's Degree in Computer Science, Information Technology, or equivalent",
    "J23": "Bachelor's degree in Business Management, Process Control Management, or similar",
    "J24": "Not Specified",
    "J25": "Diploma or Advanced/Graduate Diploma in Computer Science/IT or equivalent, with QA/Testing professional certifications",
    "J26": "Diploma or Bachelor's Degree in IT or related field",
    "J27": "Degree or Professional Qualification in Computer Science, Computer Engineering, Information Technology, or equivalent",
    "J28": "Not Specified",
    "J29": "Diploma, Advanced/Graduate Diploma, Bachelor's Degree, Post Graduate Diploma, or Professional Degree in Computer Science/IT/Engineering",
    "J30": "Bachelor's Degree, Post Graduate Diploma, Professional Degree, or Master's Degree in Engineering/Computer Science/IT or equivalent",
    "J31": "Not Specified",
    "J32": "Degree or Diploma in Computer Science, IT, or related studies",
    "J33": "BS or MS in Computer Science or related field",
    "J34": "Degree or Diploma in Computer Science",
    "J35": "Degree in Computer Science (Software Engineering), Software Engineering, IT, or equivalent",
    "J36": "Diploma or Degree in Computer Science, Information Technology, or equivalent",
    "J37": "Bachelor's degree in Computer Science or related field",
    "J38": "BS or MS in Computer Science or related field",
    "J39": "Bachelor's degree in Business Admin, Economics, Statistics, Computer Science, or related field (Master's/certs a plus)",
    "J40": "Not Specified",

    # Accounting Jobs (J41 - J60)
    "J41": "Not Specified",
    "J42": "Degree, Diploma, or Professional Certificate in Accounting or relevant disciplines",
    "J43": "Degree in Finance/Accounting, or professional qualifications (ACCA/CIMA) preferred",
    "J44": "Diploma or Bachelor's Degree in Corporate Administration/Secretarial/Management, or ICSA equivalent",
    "J45": "Bachelor's degree in Accounting & Finance, or Diploma in Business Studies (Accounting)",
    "J46": "Accounting Degree from university",
    "J47": "Not Specified",
    "J48": "SPM, STPM, or Diploma in Finance or related field",
    "J49": "Degree in Accounting and Finance, or ACCA",
    "J50": "Not Specified",
    "J51": "Bachelor's Degree in Accounting, or professional qualification (ACCA/CPA/MIA)",
    "J52": "Bachelor's degree in Accounting, Finance, or related field preferred",
    "J53": "Bachelor's degree in Accounting, Finance, or related field (Professional qualifications like ACCA/CPA/MICPA advantageous)",
    "J54": "Degree in Accounting and/or professional qualifications (MIA, ICAEW, ACCA, MICPA, CPA, CIMA)",
    "J55": "Bachelor's Degree in Accounting, Finance, or equivalent",
    "J56": "Accounting Degree Graduates, or undertaking ACCA/CIMA/MICPA/LCCI Diploma/CAT or equivalent",
    "J57": "Diploma or Degree in Accounting and/or Finance preferred",
    "J58": "Not Specified",
    "J59": "LCCI, Diploma, or Advanced Diploma in Finance/Accounting or equivalent",
    "J60": "Degree in Finance, Accounting, or equivalent",
}

# Candidate degree / qualifications from resumes
candidate_edu_specific = {
    # IT Candidates (C01 - C20)
    "C01": "Associate of Science in Business Administration",
    "C02": "Bachelor of Science in Information & Computer Science",
    "C03": "Bachelor of Science in Marketing",
    "C04": "Master of Science in Business Information Technology",
    "C05": "MBA in Management Information Systems",
    "C06": "Bachelor of Science in Information Technology",
    "C07": "Ph.D. in Organization and Management",
    "C08": "Bachelor of Science in Computer System Engineering",
    "C09": "Bachelor of Science in Business Finance",
    "C10": "Master of Arts in Counselor Education (B.A. Sociology)",
    "C11": "Bachelor of Science in Computer Information Systems (InfoSec)",
    "C12": "Master of Arts (B.S. Biology & AAS Networking)",
    "C13": "MBA & B.S. in Biology / Computer Science",
    "C14": "Bachelor of Science in Computer Engineering Technology",
    "C15": "High School Diploma",
    "C16": "Bachelor of Science in Management Information Systems",
    "C17": "Associate of Applied Science in Network Administration",
    "C18": "Associate of Science in Electronic Engineering",
    "C19": "Bachelor of Science in Information Technology Support",
    "C20": "Bachelor of Science in Sociology & Social Work Psychology",

    # Accounting Candidates (C21 - C30)
    "C21": "Bachelor of Science in Business Management",
    "C22": "Master in Accounting & Financial Management",
    "C23": "MBA in Finance",
    "C24": "Bachelor of Arts in Business Administration (Bookkeeping Certificate)",
    "C25": "Bachelor of Science in General Studies (Accounting Certificate)",
    "C26": "Professional Coursework in Graphic Design, Real Estate & Architectural Drafting",
    "C27": "Bachelor of Science in Management",
    "C28": "Bachelor of Arts in Supply Chain Management (AAS Accounting Technology)",
    "C29": "Bachelor of Science in Nursing",
    "C30": "High School Diploma in Business Management/Accounting",
}

# Helpers to format candidate and job summaries
def make_candidate_summary(row):
    cand_edu = candidate_edu_specific.get(row['candidate_id'], row['detected_education'])
    parts = [
        f"Title: {row['preferred_job_title']}",
        f"Domain: {row['category']}",
        f"Exp: {row['experience_tier']} ({row['detected_experience_years']} yrs)",
        f"Edu: {cand_edu}",
        f"Location: {row['preferred_location']}",
        f"Salary: RM {row['expected_salary_min']:,.0f}-{row['expected_salary_max']:,.0f}",
        f"Type: {row['preferred_job_type']}",
        f"Skills: {str(row.get('extracted_skills', ''))[:80]}"
    ]
    return " | ".join(parts)

def make_job_summary(row):
    sal_str = f"RM {row['salary_min']:,.0f}-{row['salary_max']:,.0f}" if row["is_salary_disclosed"] else "Undisclosed"
    edu_str = job_edu_requirements.get(row["job_id_code"], "Not Specified")
    parts = [
        f"Title: {row['job_title']}",
        f"Company: {str(row['company'])[:30]}",
        f"Domain: {row['category']}",
        f"Subcat: {row['subcategory']}",
        f"Edu: {edu_str}",
        f"Location: {row['location']}",
        f"Salary: {sal_str}",
        f"Type: {row['job_type']}"
    ]
    return " | ".join(parts)

cand_summaries = {row["candidate_id"]: make_candidate_summary(row) for _, row in resumes_df.iterrows()}
job_summaries = {row["job_id_code"]: make_job_summary(row) for _, row in jobs_df.iterrows()}

# Build annotation template rows
annotation_rows = []
for _, pair in pairs_df.iterrows():
    annotation_rows.append({
        "pair_id": pair["pair_id"],
        "candidate_id": pair["candidate_id"],
        "job_id_code": pair["job_id_code"],
        "pair_type": "",  # Blank for manual assignment
        "candidate_summary": cand_summaries.get(pair["candidate_id"], ""),
        "job_summary": job_summaries.get(pair["job_id_code"], ""),
        # Scoring columns left blank for manual scoring
        "score_title": "",
        "score_location": "",
        "score_education": "",
        "score_type": "",
        "score_salary": "",
        "score_experience": "",
        "ground_truth_score": "",
        "relevance_grade": "",
        "annotation_rationale": ""
    })

gt_template_df = pd.DataFrame(annotation_rows)

# Save template to experiments/dataset/processed/benchmark_ground_truth.csv
output_path = PROCESSED_DIR / "benchmark_ground_truth.csv"
gt_template_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Exported annotation template: {output_path} ({gt_template_df.shape[0]} rows, {gt_template_df.shape[1]} columns)")


Exported annotation template: c:\Users\yzhoo\AI-Agent-Job-Search\experiments\dataset\processed\benchmark_ground_truth.csv (120 rows, 15 columns)


In [4]:
print("-" * 65)
print("Checking Ground Truth Template")
print("-" * 65)

# 1. Row count
assert len(gt_template_df) == 120, f"Expected 120 rows, got {len(gt_template_df)}"
print(f"Row count: {len(gt_template_df)} pairs")

# 2. Check for duplicates
dup_count = gt_template_df.duplicated(subset=["candidate_id", "job_id_code"]).sum()
assert dup_count == 0, f"Found {dup_count} duplicate pairs"
print("No duplicate pairs found")

# 3. Check candidate pair counts
unique_cands = gt_template_df["candidate_id"].nunique()
assert unique_cands == 30, f"Expected 30 unique candidates, got {unique_cands}"
cand_pair_counts = gt_template_df.groupby("candidate_id").size()
assert (cand_pair_counts == 4).all(), f"Not all candidates have 4 pairs"
print("All 30 candidates have 4 pairs")

# 4. Check that pair_type and score columns are blank for manual annotation
blank_cols = [
    "pair_type", "score_title", "score_location", "score_education", "score_type",
    "score_salary", "score_experience", "ground_truth_score",
    "relevance_grade", "annotation_rationale"
]
for col in blank_cols:
    non_blank = (gt_template_df[col] != "").sum()
    assert non_blank == 0, f"Column {col} has {non_blank} non-blank values"

# 5. Check summary texts
assert (gt_template_df["candidate_summary"].str.len() > 30).all(), "Short candidate summary detected"
assert (gt_template_df["job_summary"].str.len() > 30).all(), "Short job summary detected"

# 6. Check job coverage
unique_jobs = gt_template_df["job_id_code"].nunique()
assert unique_jobs == 60, f"Expected 60 unique jobs, got {unique_jobs}"


-----------------------------------------------------------------
Checking Ground Truth Template
-----------------------------------------------------------------
Row count: 120 pairs
No duplicate pairs found
All 30 candidates have 4 pairs


### 3. Completed Ground Truth Overview

Check the distribution of the completed file (`benchmark_ground_truth_completed.csv`) after manual scoring:
- Pair counts by type
- Relevance grade counts (0, 1, 2)


In [5]:
# Check completed ground truth file
completed_gt_path = PROCESSED_DIR / "benchmark_ground_truth_completed.csv"
assert completed_gt_path.exists(), f"File not found: {completed_gt_path}"
completed_gt_df = pd.read_csv(completed_gt_path)

print("-" * 65)
print("Completed Ground Truth Summary (120 Pairs)")
print("-" * 65)
print(f"Total evaluated pairs: {len(completed_gt_df)}")
print(f"Candidates:            {completed_gt_df['candidate_id'].nunique()} (C01 - C30)")
print(f"Jobs covered:          {completed_gt_df['job_id_code'].nunique()} / 60 (J01 - J60)")

print("\n1. Pairs by type:")
print(completed_gt_df["pair_type"].value_counts().to_string())

print("\n2. Relevance grades (0 = Irrelevant, 1 = Partially Relevant, 2 = Highly Relevant):")
grade_counts = completed_gt_df["relevance_grade"].value_counts().sort_index()
for grade, count in grade_counts.items():
    pct = count / len(completed_gt_df) * 100
    print(f"  Grade {grade}: {count:>3d} pairs ({pct:.1f}%)")

print("\n3. Ground truth score stats (0 - 100):")
print(f"  Mean:   {completed_gt_df['ground_truth_score'].mean():.2f}")
print(f"  Median: {completed_gt_df['ground_truth_score'].median():.2f}")
print(f"  Std:    {completed_gt_df['ground_truth_score'].std():.2f}")
print(f"  Min / Max: {completed_gt_df['ground_truth_score'].min():.2f} / {completed_gt_df['ground_truth_score'].max():.2f}")
print("-" * 65)


-----------------------------------------------------------------
Completed Ground Truth Summary (120 Pairs)
-----------------------------------------------------------------
Total evaluated pairs: 120
Candidates:            30 (C01 - C30)
Jobs covered:          60 / 60 (J01 - J60)

1. Pairs by type:
pair_type
Partial_Match            32
Hard_Negative            30
Cross_Domain_Negative    30
High_Match               28

2. Relevance grades (0 = Irrelevant, 1 = Partially Relevant, 2 = Highly Relevant):
  Grade 0:  73 pairs (60.8%)
  Grade 1:  27 pairs (22.5%)
  Grade 2:  20 pairs (16.7%)

3. Ground truth score stats (0 - 100):
  Mean:   47.98
  Median: 45.00
  Std:    25.53
  Min / Max: 10.00 / 90.00
-----------------------------------------------------------------
